# Paso 1: Importar Librerías y Configurar el Navegador

In [3]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from time import sleep
import csv
import os
from datetime import datetime
from random import randint
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import StaleElementReferenceException
import re

from selenium.common.exceptions import NoSuchElementException
from bs4 import BeautifulSoup, NavigableString







# Paso 2: Funciónes Generales

## Función para Iniciar Sesión en X

In [207]:
def iniciar_sesion(driver,user='', pwd='', username=''):
    driver.get('https://twitter.com/login')

    try:
        # Ingresar correo electrónico
        email_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.NAME, 'text'))
        )
        email_input.send_keys(user)
        print("✔ Correo ingresado")

        WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, '//span[normalize-space(text())="Siguiente"]'))
        ).click()

        sleep(2)

        # Verificar si se solicita "nombre de usuario o teléfono"
        try:
            # Espera de máximo 3 segundos a que aparezca otro campo de texto
            username_field = WebDriverWait(driver, 3).until(
                EC.presence_of_element_located((By.NAME, 'text'))
            )

            # Verifica que el texto del label indique "Teléfono o nombre de usuario"
            username_label = driver.find_element(By.XPATH, '//span[contains(text(), "Teléfono o nombre de usuario")]')
            if username_label and username:
                username_field.send_keys(username)
                WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.XPATH, '//span[normalize-space(text())="Siguiente"]'))
                ).click()
                print("✔ Nombre de usuario confirmado")
            else:
                print("ℹ Apareció otro campo de texto, pero no se llenó porque no se identificó como campo de usuario.")
        except TimeoutException:
            print("ℹ No se pidió confirmación adicional de nombre de usuario.")

        # Ingresar contraseña
        password_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.NAME, 'password'))
        )
        password_input.send_keys(pwd)
        print("✔ Contraseña ingresada")

        # Click en "Iniciar sesión"
        login_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[data-testid="LoginForm_Login_Button"]'))
        )
        login_button.click()
        print("✔ Se hizo clic en 'Iniciar sesión'")

        sleep(5)

    except TimeoutException as e:
        print(" Error de tiempo de espera durante el login:", e)



## Función para Navegar al Perfil de IMSS

In [208]:
def navegar_a_perfil(driver,profile_url):
    """Navega al perfil especificado."""
    driver.get(profile_url)
    sleep(5)  # Espera a que el perfil cargue

# Paso 3 Funciones Identadas

## Cargar tweets ya procesados

In [209]:
def cargar_tweets_procesados(archivo_csv):
    """Carga los IDs de tweets ya procesados desde un archivo CSV."""
    if not os.path.exists(archivo_csv):
        return set()
    with open(archivo_csv, "r", encoding="utf-8") as f:
        return set(row["tweet_id"] for row in csv.DictReader(f))


## Preparar Archivo csv


In [210]:

def preparar_archivo_csv(archivo_csv):
    """Prepara el archivo CSV para escritura, añadiendo encabezados si es necesario."""
    archivo_nuevo = not os.path.exists(archivo_csv)
    f = open(archivo_csv, "a", newline="", encoding="utf-8")
    writer = csv.DictWriter(f, fieldnames=[
        "tweet_id", "tweet_text", "comentario", "comentario_autor",
        "fecha_publicacion", "timestamp_extraccion","replies",  # ya existían
        "reposts", "likes", "views"  # NUEVAS
    ])
    if archivo_nuevo:
        writer.writeheader()
    return f, writer


## NO -> Procesar un solo tweet el original para futuras comparaciones <- NO!

In [211]:
def procesar_tweet(driver, i, tweets_procesados, writer):
    """Procesa un tweet individual: lo expande, extrae datos y guarda comentarios."""
    try:
        print(f"🔍 Buscando tweets en la página...")
        
        # 🔄 SIEMPRE re-localizar los elementos después de volver atrás
        tweets = WebDriverWait(driver, 15).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'article[data-testid="tweet"]'))
        )
        
        print(f"📊 Se encontraron {len(tweets)} tweets. Procesando tweet #{i+1}")
        
        if i >= len(tweets):
            print(f"⚠️ No hay suficientes tweets. Índice {i} fuera de rango.")
            return
        
        tweet = tweets[i]
        
        # Llevar al tweet a la vista
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", tweet)
        sleep(2)
        
        print(f"🎯 Intentando hacer clic en el tweet #{i+1}")
        
        # 🎯 Estrategias múltiples para hacer clic en el tweet
        click_exitoso = False
        
        # Estrategia 1: Buscar enlace de tiempo (más específico)
        try:
            time_link = tweet.find_element(By.CSS_SELECTOR, 'time')
            parent_link = time_link.find_element(By.XPATH, '..')
            if parent_link.tag_name == 'a':
                driver.execute_script("arguments[0].click();", parent_link)
                click_exitoso = True
                print("✅ Clic exitoso usando enlace de tiempo")
        except Exception as e:
            print(f"❌ Estrategia 1 falló: {e}")
        
        # Estrategia 2: Buscar cualquier enlace dentro del tweet
        if not click_exitoso:
            try:
                tweet_links = tweet.find_elements(By.CSS_SELECTOR, 'a[role="link"]')
                for link in tweet_links:
                    href = link.get_attribute('href')
                    if href and '/status/' in href:
                        driver.execute_script("arguments[0].click();", link)
                        click_exitoso = True
                        print("✅ Clic exitoso usando enlace de status")
                        break
            except Exception as e:
                print(f"❌ Estrategia 2 falló: {e}")
        
        # Estrategia 3: Clic directo en el tweet
        if not click_exitoso:
            try:
                driver.execute_script("arguments[0].click();", tweet)
                click_exitoso = True
                print("✅ Clic exitoso directo en el tweet")
            except Exception as e:
                print(f"❌ Estrategia 3 falló: {e}")
        
        # Estrategia 4: Clic normal de Selenium
        if not click_exitoso:
            try:
                tweet.click()
                click_exitoso = True
                print("✅ Clic exitoso con Selenium click()")
            except Exception as e:
                print(f"❌ Estrategia 4 falló: {e}")
        
        if not click_exitoso:
            print(f"❌ No se pudo hacer clic en el tweet #{i+1}")
            return
        
        # Esperar a que cargue la página del tweet individual
        print("⏳ Esperando que cargue la página del tweet...")
        try:
            WebDriverWait(driver, 15).until(
                lambda d: "/status/" in d.current_url
            )
            print("✅ Página del tweet cargada correctamente")
        except Exception as e:
            print(f"❌ Error esperando carga de página: {e}")
            print(f"URL actual: {driver.current_url}")
            return
        
        sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")

        # 🟢 Tweet principal
        tweet_element = soup.find("article", {"data-testid": "tweet"})
        if not tweet_element:
            print(f"⚠️ No se encontró el elemento del tweet principal para el tweet #{i+1}. Saltando...")
            return

        tweet_text_tag = tweet_element.find("div", {"data-testid": "tweetText"})
        tweet_text = f'"{tweet_text_tag.get_text(separator=" ", strip=True)}"' if tweet_text_tag else '"Texto no encontrado"'

        print(f"📝 Tweet extraído: {tweet_text[:50]}...")

        # ✅ Crear ID basado en texto
        tweet_id = "id_" + tweet_text.replace('"', '')[:30].strip().replace(" ", "_").replace("\n", "_")[:50]

        fecha_tag = tweet_element.find("time")
        fecha_publicacion = fecha_tag["datetime"] if fecha_tag else datetime.now().strftime("%Y-%m-%d")

        # Extraer reposts, likes, y vistas
        reposts_tag = tweet_element.find("button", {"data-testid": "retweet"})
        reposts = reposts_tag.text if reposts_tag else "0"

        likes_tag = tweet_element.find("button", {"data-testid": "like"})
        likes = likes_tag.text if likes_tag else "0"

        views_tag = tweet_element.find("a", attrs={"aria-label": lambda x: x and "views" in x})
        views = views_tag.text if views_tag else "0"

        if tweet_id in tweets_procesados:
            print(f"⚠️ Tweet {tweet_id} ya fue procesado. Saltando...")
            return

        # Marcar como procesado
        tweets_procesados.add(tweet_id)

        # 🟠 Comentarios
        print("🔍 Buscando comentarios...")
        comentarios = soup.find_all("article", {"data-testid": "tweet"})
        print(f"📊 Se encontraron {len(comentarios)} artículos (incluyendo tweet principal)")
        
        if len(comentarios) <= 1:
            # No hay comentarios, escribir una fila con el tweet solamente
            writer.writerow({
                "tweet_id": tweet_id,
                "tweet_text": tweet_text,
                "comentario": '"**VERIFIQUÉ Y NO HAY NINGÚN COMENTARIO**"',
                "comentario_autor": '"**VERIFIQUÉ Y NO HAY NINGÚN COMENTARIO**"',
                "fecha_publicacion": fecha_publicacion,
                "timestamp_extraccion": datetime.now().isoformat(),
                "reposts": reposts,
                "likes": likes,
                "views": views
            })
            print(f"✅ Tweet sin comentarios guardado: {tweet_id}")
        else:
            # Hay comentarios, procesarlos
            comentarios_guardados = 0
            for com in comentarios[1:]:
                autor_tag = com.find("div", {"data-testid": "User-Name"})
                comentario_text_tag = com.find("div", {"data-testid": "tweetText"})
                fecha_com_tag = com.find("time")

                writer.writerow({
                    "tweet_id": tweet_id,
                    "tweet_text": tweet_text,
                    "comentario": f'"{comentario_text_tag.get_text(separator=" ", strip=True)}"'
                    if comentario_text_tag else '"Comentario no encontrado"',
                    "comentario_autor": f'"{autor_tag.get_text(separator=" ", strip=True)}"'
                    if autor_tag else '"Autor desconocido"',
                    "fecha_publicacion": fecha_com_tag["datetime"] if fecha_com_tag else fecha_publicacion,
                    "timestamp_extraccion": datetime.now().isoformat(),
                    "reposts": reposts,
                    "likes": likes,
                    "views": views
                })
                comentarios_guardados += 1
            
            print(f"✅ {comentarios_guardados} comentarios guardados para el tweet {tweet_id}")

    except Exception as e:
        print(f"❌ Error detallado al procesar tweet #{i+1}: {str(e)}")
        print(f"URL actual: {driver.current_url}")
        import traceback
        traceback.print_exc()

    finally:
        # ✅ Volver atrás y esperar explícitamente a que cargue el perfil
        if "/status/" in driver.current_url:
            print("🔄 Volviendo al perfil...")
            driver.back()
            try:
                # Esperar a que la URL cambie de vuelta al perfil
                WebDriverWait(driver, 15).until(
                    lambda d: "/status/" not in d.current_url
                )
                print("✅ URL cambió de vuelta al perfil")
                
                # Esperar a que los tweets se carguen
                WebDriverWait(driver, 15).until(
                    EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'article[data-testid="tweet"]'))
                )
                sleep(2)  # Tiempo adicional para estabilizar
                print(f"🔄 Volvió correctamente al perfil después del tweet #{i+1}")
            except Exception as e:
                print(f"⚠️ Error al volver y esperar los tweets: {e}")
        else:
            print("ℹ️ No era necesario volver atrás")




## Procesar un solo tweet refactor

In [212]:
'''
def clic_en_tweet(driver,tweet):
    """Hace clic sobre el tweet usando el enlace de tiempo."""
    try:
        time_link = tweet.find_element(By.CSS_SELECTOR, 'time')
        parent_link = time_link.find_element(By.XPATH, '..')
        if parent_link.tag_name == 'a':
            driver.execute_script("arguments[0].click();", parent_link)
            return True
    except Exception:
        return False

''' 


'\ndef clic_en_tweet(driver,tweet):\n    """Hace clic sobre el tweet usando el enlace de tiempo."""\n    try:\n        time_link = tweet.find_element(By.CSS_SELECTOR, \'time\')\n        parent_link = time_link.find_element(By.XPATH, \'..\')\n        if parent_link.tag_name == \'a\':\n            driver.execute_script("arguments[0].click();", parent_link)\n            return True\n    except Exception:\n        return False\n\n'

In [213]:
def esperar_pagina_tweet(driver, timeout=15):
    """Espera hasta que la URL contenga '/status/'."""
    WebDriverWait(driver, timeout).until(lambda d: "/status/" in d.current_url)


In [214]:
import re

def extraer_datos_tweet(soup):
    """
    Devuelve (tweet_id, tweet_text, fecha, replies, reposts, likes, views).
    Ahora el regex de replies reconoce tanto '1 reply' como '4 replies'.
    """
    elem = soup.find("article", {"data-testid": "tweet"})
    
    # --- 1) Texto y ID ---
    text_tag = elem.find("div", {"data-testid": "tweetText"})
    raw_text = text_tag.get_text(" ", strip=True) if text_tag else ""
    text = re.sub(r"\s+", " ", raw_text).strip()
    tweet_id = "id_" + text[:30].replace(" ", "_")
    
    # --- 2) Fecha ---
    time_tag = elem.find("time")
    fecha = time_tag["datetime"] if (time_tag and time_tag.has_attr("datetime")) else ""
    
    # --- 3) Replies (comentarios) ---
    replies = "0"
    group = elem.find("div", {"role": "group", "aria-label": True})
    if group:
        aria = group["aria-label"]  # ej. "4 replies, 7 reposts, 29 likes, 2760 views"
        m = re.search(r"(\d+)\s+repl(?:y|ies)", aria, re.IGNORECASE)
        if m:
            replies = m.group(1)
    
    # --- 4) Reposts y Likes ---
    reposts_tag = elem.find("button", {"data-testid": "retweet"})
    raw_reposts = reposts_tag.text.strip() if reposts_tag else ""
    reposts = raw_reposts if raw_reposts.isdigit() else "0"
    
    likes_tag = elem.find("button", {"data-testid": "like"})
    raw_likes = likes_tag.text.strip() if likes_tag else ""
    likes = raw_likes if raw_likes.isdigit() else "0"
    
    # --- 5) Views ---
    views = "0"
    # Busca un span que contenga la palabra "view" o "views"
    label = elem.find(lambda t: t.name == "span" and "view" in t.get_text(strip=True).lower())
    if label:
        pt = label.parent.get_text(" ", strip=True)
        v = re.search(r"([\d,]+)\s+view", pt, re.IGNORECASE)
        if v:
            views = v.group(1).replace(",", "")
    
    return tweet_id, f'"{text}"', fecha, replies, reposts, likes, views


In [215]:
import re
from datetime import datetime

def guardar_comentarios(
    tweet_id, tweet_text, soup, writer,
    fecha_pub, replies, reposts, likes, views,
    tweet_owner_raw
):
    print(f"↪️ [guardar_comentarios] para tweet_id={tweet_id}")
    tweet_owner = tweet_owner_raw.split("·")[0].strip()

    # 1) Replies oficiales
    conv = soup.find(
        "div",
        {"role": "region", "aria-label": re.compile(r"Timeline: Conversation")}
    )
    oficiales = conv.find_all("article", {"data-testid": "tweet"}) if conv else []

    # 2) Replies ocultos bajo "spam"
    spam_cells = soup.find_all("div", {"data-testid": "cellInnerDiv"})
    spam_replies = []
    for cell in spam_cells:
        spam_replies += cell.find_all("article", {"data-testid": "tweet"})

    # 3) Todos los candidatos (sacando el original si aparece)
    candidatos = oficiales + spam_replies
    if candidatos and candidatos[0].find("time"):
        candidatos = candidatos[1:]

    print(f"🔎 oficiales: {len(oficiales)}")
    print(f"🔎 spam_cells encontradas: {len(spam_cells)}, total spam artículos: {len(spam_replies)}")
    print(f"🔎 total candidatos tras unir: {len(candidatos)}")

    # 4) Filtrar por botón reply y autor distinto
    filt = []
    for com in candidatos:
        if not com.find("button", {"data-testid": "reply"}):
            continue
        autor_tag = com.find("div", {"data-testid": "User-Name"})
        autor_norm = autor_tag.get_text(" ", strip=True).split("·")[0].strip() if autor_tag else ""
        if autor_norm != tweet_owner:
            filt.append(com)

    # ——— [4bis] Recortar a 'replies' máximo ———
    max_rep = int(replies or 0)
    auténticas = filt
    #auténticas = filt[:max_rep]
    print(f"→ auténticas tras filtrar y recortar a {len(auténticas)}/{max_rep}")

    timestamp = datetime.now().isoformat()
    def clean(txt):
        return re.sub(r"\s+", " ", txt.replace("\n"," ")).strip()

    # 5) Si no hay, guardamos genérica
    if not auténticas:
        print("⚠️ Sin respuestas auténticas: grabo genérica.")
        writer.writerow({
            "tweet_id": tweet_id,
            "tweet_text": tweet_text,
            "replies": replies,
            "comentario": '"**VERIFIQUÉ Y NO HAY NINGÚN COMENTARIO**"',
            "comentario_autor": '"**VERIFIQUÉ Y NO HAY NINGÚN COMENTARIO**"',
            "fecha_publicacion": fecha_pub,
            "timestamp_extraccion": timestamp,
            "reposts": reposts,
            "likes": likes,
            "views": views
        })
        return

    # 6) Guardar cada reply auténtica
    for idx, com in enumerate(auténticas, start=1):
        print(f"🔄 Guardando respuesta auténtica #{idx}")
        # texto
        text_div = com.find("div", {"data-testid": "tweetText"})
        comentario_raw = text_div.get_text(" ", strip=True) if text_div else "Comentario no encontrado"
        # autor
        autor_div = com.find("div", {"data-testid": "User-Name"})
        autor_raw = autor_div.get_text(" ", strip=True) if autor_div else "Autor desconocido"
        # fecha reply
        time_div = com.find("time")
        fecha_com = time_div["datetime"] if (time_div and time_div.has_attr("datetime")) else fecha_pub

        writer.writerow({
            "tweet_id": tweet_id,
            "tweet_text": tweet_text,
            "replies": replies,
            "comentario": f'"{clean(comentario_raw)}"',
            "comentario_autor": f'"{clean(autor_raw)}"',
            "fecha_publicacion": fecha_com,
            "timestamp_extraccion": timestamp,
            "reposts": reposts,
            "likes": likes,
            "views": views
        })


In [216]:
'''
def volver_al_perfil(driver, timeout=15):
    driver.back()
    WebDriverWait(driver, timeout).until(lambda d: "/status/" not in d.current_url)
    WebDriverWait(driver, timeout).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'article[data-testid="tweet"]'))
    )
''' 

'\ndef volver_al_perfil(driver, timeout=15):\n    driver.back()\n    WebDriverWait(driver, timeout).until(lambda d: "/status/" not in d.current_url)\n    WebDriverWait(driver, timeout).until(\n        EC.presence_of_all_elements_located((By.CSS_SELECTOR, \'article[data-testid="tweet"]\'))\n    )\n'

## Procesar un solo tweet función semiprincipal

In [217]:
def procesar_tweet(driver, i, tweets_procesados, writer):
    print(f"\n🚀 [procesar_tweet] Iniciando tweet #{i+1}")
    # 1) Tomamos la lista de tweets en el timeline
    tweets = WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, 'article[data-testid="tweet"]')
        )
    )
    if i >= len(tweets):
        print(f"⚠️ Índice {i} fuera de rango ({len(tweets)} tweets cargados).")
        return

    tweet = tweets[i]
    # 2) Sacamos el href del enlace de tiempo
    try:
        time_link = tweet.find_element(By.CSS_SELECTOR, 'time')
        parent_link = time_link.find_element(By.XPATH, '..')
        href = parent_link.get_attribute('href')
    except Exception:
        print(f"❌ No pude extraer el enlace del tweet #{i+1}")
        return

    # 3) Abrimos en pestaña nueva
    driver.execute_script("window.open(arguments[0], '_blank');", href)
    # Cambio al nuevo handle (última pestaña)
    driver.switch_to.window(driver.window_handles[-1])

    # 4) Espero que cargue la página del tweet
    WebDriverWait(driver, 15).until(lambda d: "/status/" in d.current_url)
    sleep(2)  # espera extra si quieres

    # 5) Parseo con BeautifulSoup
    soup = BeautifulSoup(driver.page_source, "html.parser")

    # 6) Extraigo datos
    tweet_id, tweet_text, fecha_pub, replies, reposts, likes, views = extraer_datos_tweet(soup)
    print(f"💡 tweet_id={tweet_id}  replies={replies}  reposts={reposts}  likes={likes}  views={views}")

    # 7) Control de duplicados
    if tweet_id in tweets_procesados:
        print(f"⚠️ '{tweet_id}' ya procesado, cierro pestaña y salgo.")
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        return
    tweets_procesados.add(tweet_id)

    # 8) Extraigo autor para filtrar comentarios auténticos
    owner_tag = soup.find("article", {"data-testid": "tweet"})\
                    .find("div", {"data-testid": "User-Name"})
    tweet_owner_raw = owner_tag.get_text(" ", strip=True) if owner_tag else ""

    # 9) Guardar comentarios (ahora tu función recibirá el soup completo)
    guardar_comentarios(
        tweet_id, tweet_text, soup, writer,
        fecha_pub, replies, reposts, likes, views,
        tweet_owner_raw
    )

    # 10) Cerrar esta pestaña y volver a la principal
    driver.close()
    driver.switch_to.window(driver.window_handles[0])
    print(f"🔙 [procesar_tweet] Terminado tweet #{i+1}")


# Paso 4 Funciones principales

In [218]:

def extract_text_with_emojis(tweet_div):
    """Recorre el DOM y extrae texto plano y alt de los <img> (emojis)."""
    parts = []
    for node in tweet_div.descendants:
        if isinstance(node, NavigableString):
            parts.append(str(node))
        elif node.name == "img" and node.has_attr("alt"):
            parts.append(node["alt"])
    text = " ".join(parts)
    return re.sub(r"\s+", " ", text).strip()

In [219]:

def procesar_tweet_por_url(driver, url, tweets_procesados, writer):
    print(f"\n▶️  Abriendo tweet en nueva pestaña: {url}")
    # 1) Abrir en pestaña nueva y cambiar contexto
    driver.execute_script("window.open(arguments[0], '_blank');", url)
    driver.switch_to.window(driver.window_handles[-1])
    WebDriverWait(driver, 15).until(lambda d: "/status/" in d.current_url)
    sleep(1)

    # 2) Reveal inicial de replies y posible spam
    for i in range(3):
        driver.execute_script("window.scrollBy(0, 800);")
        sleep(0.7)
    try:
        spam_btn = driver.find_element(
            By.XPATH, "//span[normalize-space(text())='Show probable spam']"
        )
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", spam_btn)
        sleep(0.5)
        spam_btn.click()
        sleep(1)
    except NoSuchElementException:
        pass

    # 3) Bucle hasta que ya no cargue más replies
    prev_count = -1
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        sleep(1)

        # Capturamos el HTML y contamos replies (artículos menos el original)
        html = driver.page_source
        soup_tmp = BeautifulSoup(html, "html.parser")
        conv = soup_tmp.find("div", {
            "role": "region",
            "aria-label": re.compile(r"Timeline: Conversation")
        })
        loaded = (len(conv.find_all("article", {"data-testid": "tweet"})) - 1) if conv else 0

        print(f"    🔄 Replies cargados: {loaded}")
        if loaded == prev_count:
            break
        prev_count = loaded

    # 4) Ya con todo cargado, parseamos el resultado final
    print(f"  🔍 Tamaño de page_source tras spam: {len(driver.page_source)}")
    soup = BeautifulSoup(driver.page_source, "html.parser")

    # 5) Extraemos datos del tweet principal
    tweet_id, tweet_text, fecha_pub, replies, reposts, likes, views = extraer_datos_tweet(soup)
    print(f"  💡 Extraído: id={tweet_id} replies={replies}")

    # 6) Evitamos duplicados
    if tweet_id in tweets_procesados:
        print("  ⚠️ ya procesado, cierro y regreso.")
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        return
    tweets_procesados.add(tweet_id)

    # 7) Capturamos autor original
    owner_tag = soup.find("article", {"data-testid": "tweet"}) \
                    .find("div", {"data-testid": "User-Name"})
    tweet_owner_raw = owner_tag.get_text(" ", strip=True) if owner_tag else ""
    print(f"  👤 Autor original: {tweet_owner_raw}")

    # 8) Guardamos todos los comentarios ya cargados
    guardar_comentarios(
        tweet_id, tweet_text, soup, writer,
        fecha_pub, replies, reposts, likes, views,
        tweet_owner_raw
    )

    # 9) Cerramos pestaña y volvemos
    print("  🔙 Cerrando pestaña y volviendo…")
    driver.close()
    driver.switch_to.window(driver.window_handles[0])


## Extraer y guardar comentarios


In [ ]:
def extraer_y_guardar_comentarios(
    driver,
    archivo_csv="comentarios_imss.csv",
    max_tweets=20,
    n_scrolls=1,       # fija aquí cuántos scrolls quieres
    scroll_pause=2
):
    wait = WebDriverWait(driver, 10)
    # entrar ya en modo “Latest”
    driver.get("https://x.com/Tu_IMSS?f=live")
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'article[data-testid="tweet"]')))

    # 1) Hacer n_scrolls scrolls sencillos
    for i in range(n_scrolls):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        print(f"🔄 Scroll fijo #{i+1}/{n_scrolls}")
        sleep(scroll_pause)

    # 2) Recolectar TODOS los enlaces (sin duplicados)
    seen = set()
    tweet_links = []
    elems = driver.find_elements(By.CSS_SELECTOR, 'article[data-testid="tweet"]')
    for t in elems:
        try:
            href = t.find_element(By.CSS_SELECTOR, 'time')\
                    .find_element(By.XPATH, '..')\
                    .get_attribute('href')
        except:
            continue
        if href and href not in seen:
            seen.add(href)
            tweet_links.append(href)
        if len(tweet_links) >= max_tweets:
            break

    print(f"🔗 Capturadas {len(tweet_links)} URLs de tweets (queríamos {max_tweets})")

    # 3) Abrir CSV y procesar cada enlace
    tweets_procesados = cargar_tweets_procesados(archivo_csv)
    f, writer = preparar_archivo_csv(archivo_csv)
    try:
        for idx, url in enumerate(tweet_links, 1):
            print(f"\n🔹 Procesando URL #{idx}/{len(tweet_links)}: {url}")
            procesar_tweet_por_url(driver, url, tweets_procesados, writer)
    finally:
        f.close()


# La princial principal

In [ ]:
try:
    # Configuración del navegador
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service)

    # Iniciar sesión
    iniciar_sesion(driver,user='jamoncayop@gmail.com', pwd='Adaptiv3@*', username='@Jaime1807816689')

    # Navegar al perfil de @Tu_IMSS
    #navegar_a_perfil(driver,'https://x.com/Tu_IMSS')
    navegar_a_perfil(driver, "https://x.com/Tu_IMSS?f=live")
    print("¡Has iniciado sesión y estás en el perfil de @Tu_IMSS!")

    # Extraer y guardar comentarios de los primeros tweets
    #extraer_y_guardar_comentarios(driver, "comentarios_imss_urls_3.csv", max_tweets=20,n_scrolls=3,scroll_pause=2 )
    extraer_y_guardar_comentarios(driver, "comentarios_imss_urls_10.csv" )

except Exception as e:
    print("❌ Error en la ejecución:", e)

finally:
    # Cerrar el navegador
    if driver:
        driver.quit()



✔ Correo ingresado
✔ Nombre de usuario confirmado
✔ Contraseña ingresada
✔ Se hizo clic en 'Iniciar sesión'
¡Has iniciado sesión y estás en el perfil de @Tu_IMSS!
🔗 Capturadas 3 URLs de tweets (queríamos 20)

🔹 Procesando URL #1/3: https://x.com/Tu_IMSS/status/1946712575560790364

▶️  Abriendo tweet en nueva pestaña: https://x.com/Tu_IMSS/status/1946712575560790364
    🔄 Replies cargados: 0
    🔄 Replies cargados: 0
  🔍 Tamaño de page_source tras spam: 422373
  💡 Extraído: id=id_¡Prepárate_para_unas_vacacione replies=0
  👤 Autor original: IMSS  @Tu_IMSS
↪️ [guardar_comentarios] para tweet_id=id_¡Prepárate_para_unas_vacacione
🔎 oficiales: 0
🔎 spam_cells encontradas: 5, total spam artículos: 2
🔎 total candidatos tras unir: 1
→ auténticas tras filtrar y recortar a 0/0
⚠️ Sin respuestas auténticas: grabo genérica.
  🔙 Cerrando pestaña y volviendo…

🔹 Procesando URL #2/3: https://x.com/Tu_IMSS/status/1946711623386714361

▶️  Abriendo tweet en nueva pestaña: https://x.com/Tu_IMSS/status/194

# ORQUESTADOR ----------------------------------------------------------------*


In [ ]:
from scraping.extractor_twitter import iniciar_sesion, navegar_a_perfil, extraer_y_guardar_comentarios
#from utils.helpers import configurar_log

# Configuración de logs
#configurar_log()

# FUNCIÓN PRINCIPAL
try:
    # Configuración del navegador
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service)

    # Iniciar sesión
    iniciar_sesion(driver,user='jamoncayop@gmail.com', pwd='Adaptiv3@*', username='@Jaime1807816689')

    # Navegar al perfil de @Tu_IMSS
    #navegar_a_perfil(driver,'https://x.com/Tu_IMSS')
    navegar_a_perfil(driver, "https://x.com/Tu_IMSS?f=live")
    print("¡Has iniciado sesión y estás en el perfil de @Tu_IMSS!")

    # Extraer y guardar comentarios de los primeros tweets
    #extraer_y_guardar_comentarios(driver, "comentarios_imss_urls_3.csv", max_tweets=20,n_scrolls=3,scroll_pause=2 )
    extraer_y_guardar_comentarios(driver, "comentarios_imss_urls_10.csv" )

except Exception as e:
    print("❌ Error en la ejecución:", e)

finally:
    # Cerrar el navegador
    if driver:
        driver.quit()


✔ Correo ingresado
✔ Nombre de usuario confirmado
✔ Contraseña ingresada
✔ Se hizo clic en 'Iniciar sesión'
¡Has iniciado sesión y estás en el perfil de @Tu_IMSS!
🔄 Scroll fijo #1/1
🔗 Capturadas 7 URLs de tweets (queríamos 20)

🔹 Procesando URL #1/7: https://x.com/Tu_IMSS/status/1946994007411716461

▶️  Abriendo tweet en nueva pestaña: https://x.com/Tu_IMSS/status/1946994007411716461
    🔄 Replies cargados: 0
    🔄 Replies cargados: 0
  🔍 Tamaño de page_source tras spam: 478348
  💡 Extraído: id=id_Mi_padre_tiene_mas_de_7_meses_ replies=0
  👤 Autor original: C B E @CHRIS_MX22 · Jul 20
↪️ [guardar_comentarios] para tweet_id=id_Mi_padre_tiene_mas_de_7_meses_
🔎 oficiales: 0
🔎 spam_cells encontradas: 11, total spam artículos: 5
🔎 total candidatos tras unir: 4
→ auténticas tras filtrar y recortar a 4/0
🔄 Guardando respuesta auténtica #1
🔄 Guardando respuesta auténtica #2
🔄 Guardando respuesta auténtica #3
🔄 Guardando respuesta auténtica #4
  🔙 Cerrando pestaña y volviendo…

🔹 Procesando URL 